<a href="https://colab.research.google.com/github/dacooooll/Imbalanced-Network-IDS-Recreation/blob/main/Data(NSL_KDD)_Loading_and_Cleaning_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Downloading The NSL-KDD Dataset Files.
!wget https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTrain+.txt
!wget https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTest+.txt


--2026-04-15 09:28:45--  https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTrain+.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19109424 (18M) [text/plain]
Saving to: ‘KDDTrain+.txt’

KDDTrain+.txt       100%[===================>]  18.22M  --.-KB/s    in 0.1s    

2026-04-15 09:28:45 (137 MB/s) - ‘KDDTrain+.txt’ saved [19109424/19109424]

--2026-04-15 09:28:45--  https://raw.githubusercontent.com/Jehuty4949/NSL_KDD/master/KDDTest+.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3441513 (3.3M) [text/plain]
Sa

In [2]:
#Checking for rows counts
!wc -l KDDTrain+.txt
!wc -l KDDTest+.txt

125973 KDDTrain+.txt
22544 KDDTest+.txt


In [3]:
import pandas as pd
import numpy as np

# Define the 43 standard NSL-KDD column names.
# (Added the final closing ] here)
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment',
    'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'attack_type', 'difficulty_level'
]

# Load the data using the provided column names.
# (Added the final closing ) here)
train_df = pd.read_csv('KDDTrain+.txt', names=columns)
test_df = pd.read_csv('KDDTest+.txt', names=columns)

# Verify the data has loaded correctly
print(f"Data Loaded Successfully!")
print(f"Training Shape: {train_df.shape}")
train_df.head()

Data Loaded Successfully!
Training Shape: (125973, 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack_type,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [4]:
# --- STEP 1: DROP METADATA ---
# The research paper focuses on 41 network features. We drop 'difficulty_level'.
train_df.drop('difficulty_level', axis=1, inplace=True)
test_df.drop('difficulty_level', axis=1, inplace=True)

# --- STEP 2: REMOVE DUPLICATES ---
# To prevent model bias, we ensure every row is unique.
train_df.drop_duplicates(inplace=True)
test_df.drop_duplicates(inplace=True)

# --- STEP 3: # Correct mapping of attack types to 5 categories (as per paper)
attack_map = {
    'normal': 'Normal',
    # DoS attacks
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'apache2': 'DoS', 'udpstorm': 'DoS',
    'processtable': 'DoS', 'worm': 'DoS', 'mailbomb': 'DoS',
    # Probe attacks
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe',
    'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',
    # R2L attacks
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'sendmail': 'R2L', 'named': 'R2L', 'snmpattack': 'R2L', 'snmpgetattack': 'R2L',
    'httptunnel': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L',
    # U2R attacks
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R',
    'rootkit': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R'
}

train_df['label'] = train_df['attack_type'].map(attack_map)
test_df['label'] = test_df['attack_type'].map(attack_map)

# Check for any unmapped attacks (will show as NaN)
print("Any unmapped attacks?", train_df['label'].isna().sum())
print("\nClass distribution:")
print(train_df['label'].value_counts())


# --- STEP 4: VERIFY CLEANUP ---
print("Cleanup Complete!")
print(f"Final Train Shape: {train_df.shape}")

Any unmapped attacks? 0

Class distribution:
label
Normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64
Cleanup Complete!
Final Train Shape: (125973, 43)


In [5]:
# --- STEP 5: DROP ATTACK_TYPE COLUMN ---
# We already extracted the information into 'label', so this column is no longer needed
train_df.drop('attack_type', axis=1, inplace=True)
test_df.drop('attack_type', axis=1, inplace=True)

print("After dropping attack_type:")
print(f"Train shape: {train_df.shape}")  # Should be (125973, 42)

# --- STEP 6: ONEHOT ENCODING ---
# Convert categorical text columns to numbers as described in the paper
# protocol_type, service, flag are the 3 categorical columns

categorical_cols = ['protocol_type', 'service', 'flag']

train_df = pd.get_dummies(train_df, columns=categorical_cols)
test_df = pd.get_dummies(test_df, columns=categorical_cols)

# Align both dataframes to have same columns
# (test set might have fewer unique values in some categories)
train_df, test_df = train_df.align(test_df, join='left', axis=1, fill_value=0)

print("\nAfter OneHot Encoding:")
print(f"Train shape: {train_df.shape}")  # Should be around (125973, 122)
print(f"Test shape: {test_df.shape}")

# --- STEP 7: VERIFY ---
print("\nFinal column count:", len(train_df.columns))
print("Sample of new columns:", list(train_df.columns[:5]))

After dropping attack_type:
Train shape: (125973, 42)

After OneHot Encoding:
Train shape: (125973, 123)
Test shape: (22544, 123)

Final column count: 123
Sample of new columns: ['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment']


In [6]:
# Save preprocessed data so other notebooks can use it
train_df.to_csv('nslkdd_train_preprocessed.csv', index=False)
test_df.to_csv('nslkdd_test_preprocessed.csv', index=False)

print("Files saved successfully!")
print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

Files saved successfully!
Train: (125973, 123)
Test:  (22544, 123)
